## Computer Vision Practice (LeNet Architecture for MNIST Dataset - Digit Detection)

#### Basic Setup

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
"""
                          CV - Computer Vision
                        -------------------------
CNN - Convolutional Neural Networks
=> Convolution Layer (Necessary Information for CNN):
   Input Image
   Image size (I)
   Number of filers/kernels
   Kernel size (n), here, n means n*n size matrix
   stride (s), here, s means, s*s size matrix. Stride is a step of kernel
   padding (p)
=> Action Function, default = tanh
=> Formula: output/Feature map = [(I - n + 2p) / s] + 1
=> Pooling Layer (extract the important features): 
   Kernel size (n)
   stride (n)
=> Apply Batch Normalization
"""

In [14]:
# select device for keeping data and model in same device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [15]:
# Hyperparameters
batch_size = 64
learning_rate = 0.001
num_epochs = 10

In [ ]:
# Preprocessing
transform = transforms.Compose([
    transforms.Pad(2), # original size 28 * 28 * 1, Padding 2 in l, r, u, d. So, new size = (2 + 28 + 2) * (2 + 28 + 2) * 1 = 32 * 32 * 1
                       # here, 1 Channel no. for Grayscal image. For, rgb, channel no. 3. i.e. (24 * 24 * 3)
                       # As we'll use LeNet-5 pretrained model, 32 * 32 size image is best for this model based on it's Architecture.
    transforms.ToTensor(), # scale down pixels values from 0 - 255 to 0 - 1
    transforms.Normalize((0.1307,), (0.3081,)) # mean, std normalization, value given by MNIST Dataset
])

#### Import and Load Dataset [Practice AMNIST dataset given by torch]

In [17]:
# import torch practice datasets
from torchvision import datasets

train_dataset = datasets.MNIST(
    root='../../../../my-practice/Dataset/amnist_data',
    train=True,
    download=True,
    transform=transform # data preprocessing of a single batch done here.
)

test_dataset = datasets.MNIST(
    root='../../../../my-practice/Dataset/amnist_data',
    train=False,
    download=True,
    transform=transform
)

In [18]:
# Load Dataset using DataLoader. DataLoader: Helps to load batch dataset. Create batches, Load data batch by batch to Dataset class for 
# preprocessed and then DataLoader provide the preprocessed data [a single batch] to our model for calculation/prediction.
from torch.utils.data import DataLoader
train_loader = DataLoader(
    dataset=train_dataset, # provide single batch [data] to the dataset for preprocessing. Later, provide the preprocessed batch to our model
    batch_size=64, # create batches with 64 data [size]
    shuffle=True # When I want to distribute randomly
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False
)

#### LeNet Architecture

In [19]:
# Details in Notes or Web serach for LeNet Architecture
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # convolution layer 1: 6 filters/Kernel, kernel/filter size(n) 5x5, stride=1(s), padding=0(p)
        self.conv1 = nn.Conv2d( # nn.Conv2d -> for image dataset, nn.Conv3d -> For video dataset
            in_channels=1, # in_channels = 1 -> grayscale image dataset, in_channels = 1 -> rgb image dataset
            out_channels=6, # output channels / Feature map size will be same as no. of filters. 6 filter extracts 6 important features
            kernel_size=5, # (n) odd number square matrix
            stride=1, # (s) step l, r, u, d for calculate feature map
            padding=0 # (p) It will not impact our resutl
        )
        # convolution layer 2: 16 filters, kernel 5x5, stride=1, padding=0
        self.conv2 = nn.Conv2d(
            in_channels=6, # from conv1, out_channels=6 will be input for conv2
            out_channels=16, # same as no. of filters
            kernel_size=5, # (n) filter/kernel size
            stride=1, # (s) steps l, r, u, d
            padding=0  # (p)
        )
        
        # pooling layer 1: kernel size 2x2,  stride=(2,2) -> If we need to reduce image size by removing unnecessary features.
        self.maxpool = nn.MaxPool2d( # MaxPool2d calculate the max value for each image patch [each ] and final result is reduce the main image. 
            kernel_size=2, # using kernel_size=2, stride=2, input image's feature will be reduce by 50% [step 2 u, d, l, r] -> reduce less important feture.
            stride=2,
        )
        
        # ANN/DFF ML Part starts here small no. of fetures value. 
        # fully connected layer 1: takes 5x5x16 -> 120
        self.fc1 = nn.Linear(in_features=5 * 5 * 16, out_features=120) # 120 individual neurons will get (5 * 5 * 16) input and each neuron will generate output (total 120)
        # fully connected layer 2: takes 120 inputs, outputs 84 values
        self.fc2 = nn.Linear(in_features=120, out_features=84) # neurons 84, input 120. so, output = no. of neurons = 84
        # fully connected layer 3: takes 84 inputs, outputs 10 values
        self.fc3 = nn.Linear(in_features=84, out_features=10) # neurons 10, input 84. so, output = no. of neurons = 10 [no. of Digits 0 - 9]
        
        # activation function
        self.relu = nn.ReLU() # take only max(0, x), less than zero [negative value] is unnecessary.

    # Calculation starts here using (Covolution Layer, Pooling layer, DFF/ANN Layer, Activation Function) and return result/prediction  
    def forward(self, x): # x is the input image which is preprocessed and we'll apply CNN, DFF and all other layers on x
        # CNN Part
        # input shape: 32 x 32 x 1
        x = self.conv1(x) # 28 x 28 x 6 -> feture map size =  [(I - n + 2p) / s] + 1 = [(32 - 5 + 2*0) / ] + 1 = 27 + 1 = 28, 
                          # 6-> no. of feature map/output which is same as the no. of Filter/Kernel. Each Filter extracts one Feature map/output. 
        x = self.maxpool(x) # 14 x 14 x 6 -> reduce 50% because of 2*2 size kernel and 2 size stride [steps] but won't change the no. of Feature map/output.
        x = self.relu(x) # 14 x 14 x 6 -> apply activation function [remove unnecessary result/features]
        
        x = self.conv2(x) # 10 x 10 x 16 -> I = 14, n = 5, s = 2, p = 0, Total filter 16. So, [(14 - 5 + 2 * 0) / 1 ] + 1 = 9 + 1 = 10.
        x = self.maxpool(x) # 5 x 5 x 16 -> reduce 50%
        x = self.relu(x) # 5 x 5 x 16 -> activation function
        
        # DFF/ANN Part
        x = x.view(-1, 5 * 5 * 16) # flattens the shape into a vector from  2D matrix[image]. As, we need to apply/feed DFF/ANN. So, we need vector for calculation. 
        
        x = self.fc1(x) # 5 * 5 * 16 input and 120 output
        x = self.relu(x)

        x = self.fc2(x) # 120 input(x) and 84 output
        x = self.relu(x) # 84
        
        x = self.fc3(x) # 84 input(x) and 10 output(no. of digits 0 - 9, final result/prediction)

        return x

In [20]:
# Define model in our device
model = LeNet5().to(device)

# Define Optimizer and Loss/Cost calculator
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.001) # used for optimize the loss/cost
criterion = nn.CrossEntropyLoss() # Used for non-binary classification

##### Model Training and Evaluation

In [21]:
for epoch in range(num_epochs):
    model.train() # train mode
    running_loss = 0.0 # loss calculate
    for images, labels in train_loader: # load images and labels [0 - 9 digits] from train loader
        images, labels = images.to(device), labels.to(device) # keep in same device
        outputs = model(images) # result/prediciton
        optimizer.zero_grad() # finding local minimum. [optimal/minimum loss points]
        loss = criterion(outputs, labels) # calculate loss
        loss.backward() # Compute Gradient Descent(Back Propagation)
        optimizer.step() # # update weights(w) and bias(b) using optimal points
        running_loss += loss.item() # sum of loss/cost
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}")

Epoch [1/10], Loss: 0.2316
Epoch [2/10], Loss: 0.0675
Epoch [3/10], Loss: 0.0479
Epoch [4/10], Loss: 0.0375
Epoch [5/10], Loss: 0.0302
Epoch [6/10], Loss: 0.0264
Epoch [7/10], Loss: 0.0231
Epoch [8/10], Loss: 0.0189
Epoch [9/10], Loss: 0.0169
Epoch [10/10], Loss: 0.0156


In [ ]:
model.eval() # evaluation mode
correct = 0
total = 0
with torch.no_grad(): # no need gradient calculation
    for images, labels in test_loader: # load from test loader
        images, labels = images.to(device), labels.to(device) # same device
        outputs = model(images) # prediction
        _, predicted = torch.max(outputs.data, 1) # each neuron has final 10 outputs probability of 0 - 9. max value is the most probability of correct predition.
        total += labels.size(0) # total prediction
        correct += (predicted == labels).sum().item() # correct preditction
print(f"Accuracy on the test set: {100 * correct / total:.2f}%")

Accuracy on the test set: 99.03%




## Tutorial: Computer Vision Fundamental and LeNet Architecture (বাংলা)

এই tutorial অংশটি notebook-এর শেষের পরে যোগ করা হলো। এখানে উপরের code block-গুলোর প্রতিটি important concept beginner learner-এর জন্য বিস্তারিতভাবে explain করা হয়েছে। লক্ষ্য হলো তুমি যেন শুধু code run না করো, বরং বুঝতে পারো image data কীভাবে CNN model-এ যায়, convolution কীভাবে feature বের করে, LeNet architecture কেন কাজ করে, এবং training/evaluation কীভাবে হয়।

---

## 1. এই notebook-এর main goal কী?

এই notebook-এর goal হলো **Computer Vision** এবং **LeNet-5 style CNN architecture** ব্যবহার করে MNIST handwritten digit classification শেখা। এখানে input image হলো grayscale digit image, আর output হলো 0 থেকে 9 পর্যন্ত digit class।

পুরো flow:

`MNIST Image -> Padding -> Tensor -> Normalize -> Convolution -> Pooling -> ReLU -> Flatten -> Fully Connected Layers -> Class Scores -> Loss -> Backpropagation -> Accuracy`

এই notebook-এ তুমি classic CNN pipeline practice করেছ। এটি computer vision শেখার জন্য খুব গুরুত্বপূর্ণ foundation।

---

## 2. Computer Vision এবং CNN কী?

**Computer Vision (CV)** হলো machine learning/deep learning-এর এমন একটি field যেখানে model image বা video data থেকে pattern বুঝতে শেখে। যেমন digit recognition, face detection, medical image classification, traffic sign recognition ইত্যাদি।

**CNN বা Convolutional Neural Network** image data-এর জন্য বিশেষভাবে effective। কারণ image-এর local pattern থাকে: edge, line, curve, texture, shape। CNN ছোট ছোট filter/kernel দিয়ে image scan করে এই pattern বের করে।

### CNN-এর key parts

- `Convolution Layer`: image থেকে feature map বের করে
- `Activation Function`: non-linearity যোগ করে
- `Pooling Layer`: feature map ছোট করে, important feature রাখে
- `Flatten`: 2D/3D feature map কে vector বানায়
- `Fully Connected Layer`: final classification করে

---

## 3. Convolution Output Size Formula

Notebook-এ formula দেওয়া আছে:

\[Output = \frac{I - K + 2P}{S} + 1\]

এখানে:

- `I` = input image/feature map size
- `K` = kernel/filter size
- `P` = padding
- `S` = stride

### Example

Input size 32, kernel 5, padding 0, stride 1 হলে:

\[Output = \frac{32 - 5 + 2(0)}{1} + 1 = 28\]

তাই `32x32` image convolution-এর পরে `28x28` feature map হয়।

---

## 4. Cell-by-Cell Explanation

### Cell 0: Notebook Title
`Computer Vision Practice (LeNet Architecture for AMNIST Dataset - Digit Detection)`

এখানে notebook-এর topic বলা হয়েছে। Note: code-এ `datasets.MNIST` ব্যবহার করা হয়েছে, তাই এটি MNIST dataset। Title/comment-এ AMNIST লেখা থাকলেও actual dataset MNIST।

### Cell 1-2: Basic Setup and Imports

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
```

এখানে deep learning এবং computer vision-এর জন্য প্রয়োজনীয় package import করা হয়েছে।

- `torch`: tensor operation এবং model training
- `torch.nn`: neural network layer, model, loss
- `torch.nn.functional`: functional API, যদিও এই notebook-এ `F` ব্যবহার হয়নি
- `torch.optim`: optimizer, যেমন Adam
- `torchvision.datasets`: MNIST dataset load করার জন্য
- `torchvision.transforms`: preprocessing করার জন্য
- `DataLoader`: batch-wise data loading করার জন্য

### Cell 3: CNN Notes

এই cell-এ CNN-এর core concepts note আকারে লেখা হয়েছে: input image, filter/kernel, kernel size, stride, padding, activation, pooling, batch normalization।

এটি beginner-এর জন্য ভালো summary। তবে LeNet original architecture-এ activation হিসেবে historically `tanh` ছিল, কিন্তু এই notebook-এ practical modern choice হিসেবে `ReLU` ব্যবহার করা হয়েছে।

### Cell 4: Device Selection

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

এখানে model CPU নাকি GPU-তে run করবে তা select করা হয়েছে। GPU থাকলে `cuda`, না থাকলে `cpu`। PyTorch-এ model এবং data একই device-এ থাকতে হয়, না হলে runtime error হবে।

### Cell 5: Hyperparameters

```python
batch_size = 64
learning_rate = 0.001
num_epochs = 10
```

Hyperparameter হলো training শুরু করার আগে manually set করা value।

- `batch_size=64`: একবারে 64টি image model-এ যাবে
- `learning_rate=0.001`: optimizer কত বড় step নেবে
- `num_epochs=10`: পুরো training dataset 10 বার model দেখবে

Learning rate খুব important। বেশি হলে loss unstable হতে পারে, কম হলে training slow হতে পারে।


## 5. Preprocessing, Dataset, DataLoader, and LeNet Architecture

### Cell 6: Preprocessing Transform

```python
transform = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
```

এখানে image preprocessing pipeline তৈরি করা হয়েছে।

#### `transforms.Pad(2)`

MNIST image original size `28x28x1`। Padding 2 দিলে চার পাশে 2 pixel করে যোগ হয়।

\[28 + 2 + 2 = 32\]

তাই image হয় `32x32x1`। LeNet-5 original architecture 32x32 image input ধরে design করা হয়েছিল, তাই padding করা হয়েছে।

#### `transforms.ToTensor()`

Pixel value 0-255 থেকে 0-1 range-এ scale করে।

\[x_{scaled} = \frac{x}{255}\]

#### `transforms.Normalize((0.1307,), (0.3081,))`

Normalize formula:

\[x_{norm} = \frac{x - mean}{std}\]

MNIST dataset-এর commonly used mean `0.1307` এবং std `0.3081`। Normalization training stable করতে সাহায্য করে।

### Cell 7-8: Dataset Load

```python
train_dataset = datasets.MNIST(..., train=True, transform=transform)
test_dataset = datasets.MNIST(..., train=False, transform=transform)
```

এখানে training এবং test dataset load করা হয়েছে। `transform` apply হওয়ায় image load হওয়ার সময় preprocessing automatically হবে।

### Cell 9: DataLoader

```python
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)
```

DataLoader dataset থেকে mini-batch তৈরি করে।

- training-এ `shuffle=True`: data random order-এ যায়, model order memorize করে না
- test-এ `shuffle=False`: evaluation-এর জন্য order জরুরি না

### Cell 10-11: LeNet5 Model Class

এই cell notebook-এর সবচেয়ে important অংশ। এখানে CNN architecture define করা হয়েছে।

### LeNet shape flow

Input image shape:

`1 x 32 x 32`

Forward pass-এর shape flow:

`1x32x32 -> Conv1 -> 6x28x28 -> MaxPool -> 6x14x14 -> Conv2 -> 16x10x10 -> MaxPool -> 16x5x5 -> Flatten -> 400 -> FC1 120 -> FC2 84 -> FC3 10`

### `conv1 = nn.Conv2d(1, 6, kernel_size=5)`

প্রথম convolution layer।

- `in_channels=1`: grayscale image
- `out_channels=6`: 6টি filter, তাই 6টি feature map
- `kernel_size=5`: প্রতিটি filter 5x5
- `stride=1`, `padding=0`

Output size:

\[\frac{32 - 5 + 2(0)}{1} + 1 = 28\]

তাই output: `6 x 28 x 28`।

### `maxpool = nn.MaxPool2d(kernel_size=2, stride=2)`

Max pooling 2x2 patch থেকে maximum value নেয়। এটি feature map size অর্ধেক করে।

`6x28x28 -> 6x14x14`

Pooling-এর use case:

- spatial size কমানো
- computation কমানো
- important feature রাখা
- ছোট translation change-এ model একটু robust করা

### `ReLU` activation

Formula:

\[ReLU(x) = max(0, x)\]

Negative value zero করে, positive value রাখে। এটি non-linearity যোগ করে।

### `conv2 = nn.Conv2d(6, 16, kernel_size=5)`

দ্বিতীয় convolution layer। Conv1 থেকে 6 channel আসে, Conv2 16টি feature map বানায়।

Input size `14x14`, kernel `5`, stride `1`, padding `0`।

\[\frac{14 - 5 + 2(0)}{1} + 1 = 10\]

Output: `16 x 10 x 10`। তারপর maxpool করলে `16 x 5 x 5`।

### Flatten

```python
x = x.view(-1, 5 * 5 * 16)
```

CNN feature map কে fully connected layer-এ দেওয়ার আগে vector বানাতে হয়।

\[16 \times 5 \times 5 = 400\]

তাই flattened feature size = 400।

### Fully Connected Layers

- `fc1`: 400 -> 120
- `fc2`: 120 -> 84
- `fc3`: 84 -> 10

শেষ layer 10 output দেয়, কারণ digit class 0 থেকে 9 পর্যন্ত মোট 10টি। এগুলো raw class scores বা logits।


## 6. Model Setup, Training, Evaluation, and Improvements

### Cell 12: Model, Optimizer, Loss

```python
model = LeNet5().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
```

এখানে model তৈরি করে device-এ পাঠানো হয়েছে। তারপর optimizer এবং loss function define করা হয়েছে।

### `CrossEntropyLoss` কেন?

MNIST multi-class classification problem। এখানে 10টি class আছে। `CrossEntropyLoss` logits নেয় এবং internally softmax + negative log likelihood apply করে।

Softmax formula:

\[P(y=i) = \frac{e^{z_i}}{\sum_j e^{z_j}}\]

তারপর true class probability যত কম হবে, loss তত বেশি হবে।

### `Adam` optimizer

Adam gradient ব্যবহার করে weight update করে। Conceptual update:

\[W_{new} = W_{old} - \eta \frac{\partial L}{\partial W}\]

এখানে `eta` বা `learning_rate = 0.001`।

### Cell 13-14: Model Training

Training loop-এর main steps:

1. `model.train()` দিয়ে training mode চালু করা
2. `images, labels = images.to(device), labels.to(device)` দিয়ে data same device-এ নেওয়া
3. `outputs = model(images)` দিয়ে forward pass করা
4. `loss = criterion(outputs, labels)` দিয়ে loss calculate করা
5. `optimizer.zero_grad()` দিয়ে old gradient clear করা
6. `loss.backward()` দিয়ে backpropagation করা
7. `optimizer.step()` দিয়ে weight update করা
8. running loss জমা করে epoch loss print করা

### Backpropagation intuition

Loss থেকে model বুঝে কোন weight কতটা ভুলের জন্য responsible। তারপর gradient calculate করে optimizer weight update করে।

### Cell 15: Evaluation

```python
model.eval()
with torch.no_grad():
    ...
```

Evaluation-এর সময় gradient দরকার নেই, তাই `torch.no_grad()` ব্যবহার করা হয়েছে। এতে memory কম লাগে এবং inference দ্রুত হয়।

Prediction বের করার line:

```python
_, predicted = torch.max(outputs.data, 1)
```

প্রতিটি image-এর জন্য 10টি class score আসে। সবচেয়ে বড় score-এর index predicted digit।

Accuracy formula:

\[Accuracy = \frac{Correct\ Predictions}{Total\ Predictions} \times 100\]

---

## 7. Topics You Missed or Can Improve

এই notebook ভালো CNN foundation practice, কিন্তু আরও professional এবং complete করতে নিচের improvement দরকার।

### 1. Dataset title/comment consistency
Notebook title/comment-এ `AMNIST` লেখা আছে, কিন্তু code-এ `datasets.MNIST` ব্যবহার করা হয়েছে। Beginner হিসেবে dataset name consistent রাখা ভালো।

### 2. Validation set নেই
Training শেষে সরাসরি test set evaluate করা হয়েছে। Better practice হলো:

- train set: model শেখার জন্য
- validation set: epoch-wise tuning/checking
- test set: final unbiased performance measure

### 3. Training loss curve নেই
Epoch-wise loss plot করলে বোঝা যেত model stable ভাবে শিখছে কি না।

### 4. Confusion matrix নেই
Accuracy শুধু overall score দেয়। Confusion matrix দিলে বোঝা যেত model কোন digit বেশি ভুল করছে, যেমন 3 কে 8 ভাবছে কি না।

### 5. Sample predictions visualize করা হয়নি
কিছু test image দেখিয়ে actual vs predicted label দেখালে model behaviour আরও পরিষ্কার হতো।

### 6. Model checkpoint নেই
`torch.save(model.state_dict(), path)` দিয়ে trained model save করা উচিত। এতে পরে reuse/inference করা যাবে।

### 7. Parameter count নেই
LeNet-এর total trainable parameter count করলে architecture complexity বোঝা যেত।

### 8. `F` import unused
`torch.nn.functional as F` import করা হয়েছে কিন্তু ব্যবহার করা হয়নি। ব্যবহার না করলে remove করা যায়, বা `F.relu()` দিয়ে functional style দেখানো যায়।

### 9. Batch Normalization note আছে, implementation নেই
Initial note-এ BatchNorm mention আছে, কিন্তু LeNet class-এ BatchNorm layer নেই। চাইলে `nn.BatchNorm2d(6)` এবং `nn.BatchNorm2d(16)` conv layer-এর পরে add করে experiment করা যায়।

### 10. Original LeNet vs modern LeNet difference
Original LeNet-5 সাধারণত tanh/sigmoid style activation এবং average pooling ব্যবহার করত। এই notebook modernized version হিসেবে ReLU এবং MaxPool ব্যবহার করেছে। এটা practical, কিন্তু difference জানা important।

---

## 8. Beginner Big Picture

এই notebook-কে সহজভাবে এভাবে ভাবো:

1. Digit image load করা হলো
2. Image 28x28 থেকে padding দিয়ে 32x32 করা হলো
3. Tensor এবং normalized input বানানো হলো
4. Conv1 edge/simple feature বের করল
5. Pooling feature map ছোট করল
6. Conv2 আরও complex feature বের করল
7. আবার pooling করে compact representation বানানো হলো
8. Flatten করে vector বানানো হলো
9. Fully connected layer final digit class score দিল
10. CrossEntropyLoss ভুল মাপল
11. Adam optimizer weight update করল
12. Test set-এ accuracy বের হলো

---

## 9. Final Takeaway

এই notebook computer vision-এর core foundation শেখায়: convolution, kernel, stride, padding, feature map, pooling, flattening, fully connected classifier, CrossEntropy loss, optimizer, training loop, এবং evaluation।

যদি তুমি এই notebook ভালোভাবে বুঝে ফেলো, তাহলে next step হওয়া উচিত: validation split, confusion matrix, prediction visualization, model save/load, BatchNorm experiment, এবং তারপর modern CNN architectures যেমন AlexNet, VGG, ResNet শেখা।
